# Copula-Based CF for Non-Gaussian Distributions

**Circulatory Fidelity: Quantifying Structural Coupling to Diagnose Mean-Field Failure**

This notebook demonstrates the Gaussian copula transform for computing CF on non-Gaussian continuous data.

---

## Estimation Methods

| Distribution Type | Method | Properties |
|-------------------|--------|------------|
| Gaussian | Closed-form $r_L = |\rho|$ | Exact |
| Non-Gaussian continuous | **Copula transform** | Conservative lower bound, closed-form SE |
| Discrete/mixed | KSG (k-NN) | Use with caution (30-45% bias) |

The copula transform is **recommended** for all continuous distributions because:
1. It provides a principled conservative (lower bound) estimate
2. It has closed-form standard errors via the Fisher transformation
3. It avoids the substantial negative bias of k-NN estimators

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['font.size'] = 11
np.random.seed(42)

SIGMA_MIN = 1.0 / np.sqrt(2 * np.pi * np.e)

## Gaussian CF (Closed-Form Reference)

In [ ]:
def mutual_information_gaussian(rho: float) -> float:
    """MI for bivariate Gaussian: I = -0.5 * log(1 - rho^2)"""
    rho = np.clip(rho, -0.9999, 0.9999)
    return -0.5 * np.log(1 - rho**2)

def differential_entropy_gaussian(sigma: float) -> float:
    """Entropy for univariate Gaussian: H = 0.5 * log(2*pi*e*sigma^2)"""
    return 0.5 * np.log(2 * np.pi * np.e * sigma**2)

def cf_gaussian(rho: float, sigma_z: float, sigma_x: float) -> float:
    """Closed-form CF for bivariate Gaussian."""
    mi = mutual_information_gaussian(rho)
    h_min = min(differential_entropy_gaussian(sigma_z), differential_entropy_gaussian(sigma_x))
    if h_min <= 0:
        return np.nan
    return np.clip(mi / h_min, 0.0, 1.0)

## Copula-Based Estimation

The key insight: **mutual information is invariant under monotonic transformations**.

For continuous random variables $Z$ and $X$:
$$I(Z; X) = I(F_Z(Z); F_X(X)) = I(\Phi^{-1}(U); \Phi^{-1}(V))$$

where $U = F_Z(Z)$ and $V = F_X(X)$ are the probability integral transforms.

### The Gaussian Copula Minimum Entropy Theorem

Among all copulas with correlation $\rho$, the Gaussian copula **minimizes** mutual information (Joe, 1989):
$$I_{\text{Gauss}}(\rho) = \min_{C: \text{Corr}_C = \rho} I_C$$

This means our estimate is a **conservative lower bound** - if $\widehat{\text{CF}} > \tau$, the true CF is at least as large.

In [ ]:
def mutual_information_copula(x: np.ndarray, y: np.ndarray) -> float:
    """
    Estimate MI using Gaussian copula transform.
    
    Returns a CONSERVATIVE LOWER BOUND on true MI.
    """
    n = len(x)
    
    # Rank transform to uniform (0, 1)
    u = (stats.rankdata(x) - 0.5) / n
    v = (stats.rankdata(y) - 0.5) / n
    
    # Transform to standard normal
    z = stats.norm.ppf(u)
    w = stats.norm.ppf(v)
    
    # Copula correlation
    rho_c = np.corrcoef(z, w)[0, 1]
    
    # Closed-form MI for Gaussian copula
    rho_c = np.clip(rho_c, -0.9999, 0.9999)
    return -0.5 * np.log(1 - rho_c**2)


def cf_copula(x: np.ndarray, y: np.ndarray) -> float:
    """
    Compute CF using Gaussian copula transform.
    
    RECOMMENDED for non-Gaussian continuous distributions.
    """
    mi = mutual_information_copula(x, y)
    
    # Standard normal entropy = 0.5 * log(2*pi*e)
    h = 0.5 * np.log(2 * np.pi * np.e)
    
    return np.clip(mi / h, 0.0, 1.0)


def copula_correlation_with_ci(x: np.ndarray, y: np.ndarray):
    """
    Compute copula correlation with Fisher standard error and 95% CI.
    """
    n = len(x)
    
    # Rank transform
    u = (stats.rankdata(x) - 0.5) / n
    v = (stats.rankdata(y) - 0.5) / n
    
    # Normal transform
    z = stats.norm.ppf(u)
    w = stats.norm.ppf(v)
    
    # Copula correlation
    rho_c = np.corrcoef(z, w)[0, 1]
    
    # Fisher standard error
    se = 1.0 / np.sqrt(n - 3) if n > 3 else np.inf
    
    # 95% CI via Fisher transformation
    z_transform = np.arctanh(rho_c)
    ci_z = (z_transform - 1.96 * se, z_transform + 1.96 * se)
    ci_95 = (np.tanh(ci_z[0]), np.tanh(ci_z[1]))
    
    return rho_c, se, ci_95

## Validation 1: Gaussian Data (Exactness)

In [ ]:
print("Gaussian Validation: Copula should match closed-form")
print("="*60)

n_samples = 5000
rho_values = [0.0, 0.3, 0.5, 0.7, 0.9]

for rho in rho_values:
    # Generate bivariate Gaussian
    cov = [[1, rho], [rho, 1]]
    data = np.random.multivariate_normal([0, 0], cov, n_samples)
    x, y = data[:, 0], data[:, 1]
    
    # True MI
    true_mi = mutual_information_gaussian(rho)
    
    # Copula estimate
    copula_mi = mutual_information_copula(x, y)
    
    # Pearson estimate (for comparison)
    pearson_rho = np.corrcoef(x, y)[0, 1]
    pearson_mi = mutual_information_gaussian(pearson_rho)
    
    print(f"rho={rho:.1f}: True MI={true_mi:.4f}, Copula={copula_mi:.4f}, Pearson={pearson_mi:.4f}")

## Validation 2: Log-Normal Marginals (Copula Invariance)

If $(Z, X)$ is bivariate Gaussian with correlation $\rho$, then $(e^Z, e^X)$ has the same MI.

- **Pearson correlation will fail** (it measures linear association in the transformed space)
- **Copula transform should recover the true MI** (because MI is invariant)

In [ ]:
print("Log-Normal Validation: Copula should be exact, Pearson should fail")
print("="*60)

for rho in [0.3, 0.5, 0.7, 0.9]:
    # Generate bivariate Gaussian, then exponentiate
    cov = [[1, rho], [rho, 1]]
    data = np.random.multivariate_normal([0, 0], cov, n_samples)
    x, y = np.exp(data[:, 0]), np.exp(data[:, 1])  # Log-normal transform
    
    # True MI (same as Gaussian!)
    true_mi = mutual_information_gaussian(rho)
    
    # Copula estimate
    copula_mi = mutual_information_copula(x, y)
    
    # Pearson estimate (WRONG for log-normal)
    pearson_rho = np.corrcoef(x, y)[0, 1]
    pearson_mi = mutual_information_gaussian(pearson_rho)
    
    pearson_error = 100 * (pearson_mi - true_mi) / true_mi
    copula_error = 100 * (copula_mi - true_mi) / true_mi
    
    print(f"rho={rho:.1f}: True MI={true_mi:.4f}, Copula={copula_mi:.4f} ({copula_error:+.1f}%), "
          f"Pearson={pearson_mi:.4f} ({pearson_error:+.1f}%)")

## Validation 3: Student-t Distribution (Conservative Bound)

The Student-t copula has heavier tails than Gaussian, resulting in **higher** MI for the same correlation.

The Gaussian copula estimate should be a **conservative lower bound**.

In [ ]:
print("Student-t Validation: Copula should underestimate (conservative)")
print("="*60)

rho = 0.7
gaussian_mi = mutual_information_gaussian(rho)

for nu in [3, 5, 10, 30, 100]:
    # Generate correlated Student-t via Gaussian copula + t marginals
    cov = [[1, rho], [rho, 1]]
    data = np.random.multivariate_normal([0, 0], cov, n_samples)
    u = stats.norm.cdf(data)
    x = stats.t.ppf(u[:, 0], nu)
    y = stats.t.ppf(u[:, 1], nu)
    
    copula_mi = mutual_information_copula(x, y)
    underestimation = 100 * (gaussian_mi - copula_mi) / gaussian_mi
    
    print(f"nu={nu:3d}: Copula MI={copula_mi:.4f}, Gaussian bound={gaussian_mi:.4f}, "
          f"Underestimation={underestimation:.1f}%")

## Validation 4: Non-Monotonic Dependence

The copula transform assumes **monotonic** dependence. For non-monotonic relationships (V-shaped, circular), it returns ~0.

**This is correct behavior**: low pairwise CF triggers Stage 2 of the two-stage protocol.

In [ ]:
print("Non-Monotonic Validation: Copula should return ~0 (triggers Stage 2)")
print("="*60)

for strength in [0.5, 1.0, 2.0]:
    # V-shaped dependence: y = |x| + noise
    x = np.random.randn(n_samples)
    y = strength * np.abs(x) + 0.5 * np.random.randn(n_samples)
    
    copula_mi = mutual_information_copula(x, y)
    pearson_rho = np.corrcoef(x, y)[0, 1]
    
    print(f"Strength={strength:.1f}: Copula MI={copula_mi:.4f}, Pearson rho={pearson_rho:.4f}")

print("\n--> Low copula MI with clear dependence = Stage 2 triggered!")

## Comparison: Copula vs KSG

The table below summarizes why we prefer copula-based estimation:

In [ ]:
comparison_data = {
    'Property': ['Bias (moderate ρ)', 'Bias (zero ρ)', 'Standard errors', 
                 'Sample size for CV<10%', 'Dimension scaling', 'Non-monotonic detection'],
    'KSG (k-NN)': ['-30 to -45%', 'Positive', 'Not available', 
                  'N > 5,000', 'Fails for d > 2', 'Poor'],
    'Copula Transform': ['< 1% (exact or conservative)', 'Zero', 'Closed-form (Fisher)',
                        'N ≈ 500', 'N/A (scalar pairs)', 'Triggers Stage 2']
}

print("Comparison: Copula Transform vs KSG")
print("="*80)
print(f"{'Property':<30} {'KSG (k-NN)':<25} {'Copula Transform':<25}")
print("-"*80)
for i, prop in enumerate(comparison_data['Property']):
    print(f"{prop:<30} {comparison_data['KSG (k-NN)'][i]:<25} {comparison_data['Copula Transform'][i]:<25}")

## Practical Workflow

```
1. Is your data Gaussian?
   → YES: Use closed-form r_L = |ρ| (exact)
   → NO: Continue to step 2

2. Is your data continuous?
   → YES: Use copula transform (conservative, closed-form SE)
   → NO (discrete/mixed): Use KSG with awareness of bias

3. Is CF ≈ 0 but you suspect dependence?
   → Apply Stage 2: Check interaction CF
```

## Summary

The Gaussian copula transform:

1. **Is exact** for Gaussian copulas (includes all elliptical distributions after marginal transformation)
2. **Provides conservative lower bounds** for non-Gaussian copulas
3. **Has closed-form standard errors** via the Fisher transformation
4. **Correctly returns ~0 for non-monotonic dependence**, triggering the two-stage protocol

Use KSG only for discrete or mixed discrete-continuous variables where rank transforms are undefined.